In [ ]:
import pandas as pd
import numpy as np

def pick_col(df, candidates):
    """Return the first candidate column that exists in df, else None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_sex_to_01(series: pd.Series) -> pd.Series:
    """
    Normalize various sex/gender encodings to 0=female, 1=male.
    Handles:
      - PPMI-style: 0/1 already
      - HIVE-style: 1=male, 2=female
      - strings like 'M','F','Male','Female'
    Returns Int64 (nullable).
    """
    s = series.astype("string").str.strip().str.lower()

    # Try numeric first
    num = pd.to_numeric(s, errors="coerce")

    out = pd.Series(pd.array([pd.NA] * len(s), dtype="Int64"), index=s.index)

    # Case A: already 0/1
    out = out.mask(num.isin([0, 1]), num.where(num.isin([0, 1])).astype("Int64"))

    # Case B: 1/2 with 1=male,2=female
    map_12 = num.map({1: 1, 2: 0})
    out = out.fillna(map_12.astype("Int64"))

    # Case C: strings
    str_map = {
        "m": 1, "male": 1, "man": 1,
        "f": 0, "female": 0, "woman": 0
    }
    out = out.fillna(s.map(str_map).astype("Int64"))

    return out

def compare_ppmi_hive_identity(
    PPMI_df: pd.DataFrame,
    HIVE_df: pd.DataFrame,
    id_col="_id",
    # set these to override autodetection if you want:
    ppmi_age=None, ppmi_sex=None, ppmi_race=None,
    hive_age=None, hive_sex=None, hive_race=None,
    age_tol=1
):
    # --- autodetect columns if not provided ---
    if ppmi_age is None:
        ppmi_age = pick_col(PPMI_df, ["age", "age_at_visit", "AGE"])
    if ppmi_sex is None:
        ppmi_sex = pick_col(PPMI_df, ["SEX", "sex", "gender"])
    if ppmi_race is None:
        ppmi_race = pick_col(PPMI_df, ["race", "RACE", "ETHNICITY"])

    if hive_age is None:
        hive_age = pick_col(HIVE_df, ["age", "AGE"])
    if hive_sex is None:
        hive_sex = pick_col(HIVE_df, ["gen", "gender", "SEX", "sex"])
    if hive_race is None:
        hive_race = pick_col(HIVE_df, ["race", "RACE", "ethnicity"])

    missing = [x for x in [ppmi_age, ppmi_sex, ppmi_race, hive_age, hive_sex, hive_race] if x is None]
    if missing:
        raise ValueError(
            "Could not find all required columns automatically. "
            f"Detected: ppmi_age={ppmi_age}, ppmi_sex={ppmi_sex}, ppmi_race={ppmi_race}, "
            f"hive_age={hive_age}, hive_sex={hive_sex}, hive_race={hive_race}. "
            "Pass explicit column names to compare_ppmi_hive_identity(...)."
        )

    # --- 1) merge one-to-one on id ---
    check = (
        PPMI_df[[id_col, ppmi_age, ppmi_sex, ppmi_race]]
        .merge(
            HIVE_df[[id_col, hive_age, hive_sex, hive_race]],
            on=id_col,
            how="inner",
            suffixes=("_ppmi", "_hive"),
            validate="one_to_one"
        )
        .rename(columns={
            ppmi_age: "age_ppmi",
            ppmi_sex: "sex_ppmi_raw",
            ppmi_race: "race_ppmi_raw",
            hive_age: "age_hive",
            hive_sex: "sex_hive_raw",
            hive_race: "race_hive_raw",
        })
    )

    # --- 2) numeric coercions (safe even if you already did some) ---
    check["age_ppmi"] = pd.to_numeric(check["age_ppmi"], errors="coerce")
    check["age_hive"] = pd.to_numeric(check["age_hive"], errors="coerce")

    check["sex_ppmi_n"] = normalize_sex_to_01(check["sex_ppmi_raw"])
    check["sex_hive_n"] = normalize_sex_to_01(check["sex_hive_raw"])

    # race: keep numeric codes if possible, else keep as string for comparison
    race_ppmi_num = pd.to_numeric(check["race_ppmi_raw"], errors="coerce")
    race_hive_num = pd.to_numeric(check["race_hive_raw"], errors="coerce")

    # If both mostly numeric, compare numerically; otherwise compare normalized strings
    if race_ppmi_num.notna().mean() > 0.8 and race_hive_num.notna().mean() > 0.8:
        check["race_ppmi_n"] = race_ppmi_num.astype("Int64")
        check["race_hive_n"] = race_hive_num.astype("Int64")
    else:
        check["race_ppmi_n"] = check["race_ppmi_raw"].astype("string").str.strip().str.lower()
        check["race_hive_n"] = check["race_hive_raw"].astype("string").str.strip().str.lower()

    # --- 3) keep rows with all three features present (per dataset) ---
    non_missing = check[
        check["age_ppmi"].notna() &
        check["age_hive"].notna() &
        check["sex_ppmi_n"].notna() &
        check["sex_hive_n"].notna() &
        check["race_ppmi_n"].notna() &
        check["race_hive_n"].notna()
    ].copy()

    # --- 4) mismatch flags ---
    non_missing["age_mismatch"] = (non_missing["age_ppmi"] - non_missing["age_hive"]).abs() > age_tol
    non_missing["sex_mismatch"] = non_missing["sex_ppmi_n"] != non_missing["sex_hive_n"]
    non_missing["race_mismatch"] = non_missing["race_ppmi_n"] != non_missing["race_hive_n"]

    # summary
    summary = {
        "merged_rows": len(check),
        "rows_with_all_3_present": len(non_missing),
        "age_mismatches": int(non_missing["age_mismatch"].sum()),
        "sex_mismatches": int(non_missing["sex_mismatch"].sum()),
        "race_mismatches": int(non_missing["race_mismatch"].sum()),
    }

    # helpful view: only the problematic rows
    mismatches = non_missing[
        non_missing["age_mismatch"] | non_missing["sex_mismatch"] | non_missing["race_mismatch"]
    ][
        [id_col,
         "age_ppmi", "age_hive", "age_mismatch",
         "sex_ppmi_raw", "sex_hive_raw", "sex_ppmi_n", "sex_hive_n", "sex_mismatch",
         "race_ppmi_raw", "race_hive_raw", "race_ppmi_n", "race_hive_n", "race_mismatch"]
    ].sort_values([ "age_mismatch", "sex_mismatch", "race_mismatch" ], ascending=False)

    return check, non_missing, mismatches, summary


# --- run it on your current matched frames ---
check, check_non_missing, mismatches, summary = compare_ppmi_hive_identity(
    PPMI_keep, hive_keep,
    id_col="_id",
    age_tol=1
)

print(summary)
display(mismatches.head(20))


In [ ]:
"""#1. Align rows explicitly by _id
#Creating a comparison table where each row is one subject

check = (
    PPMI_keep[["_id", "age", "SEX", "race"]]
    .merge(
        hive_keep[["_id", "age", "gen", "race"]],
        on="_id",
        how="inner",
        suffixes=("_ppmi", "_hive"),
        validate="one_to_one"
    )
)

#one_to_one is important, if something is wrong it will fail

#2. Keeping only rows containing all three features (no missing values)

check_non_missing = check[
    check["age_ppmi"].notna() &
    check["age_hive"].notna() &
    check["SEX"].notna() &
    check["race_ppmi"].notna() &
    check["race_hive"].notna()
].copy()

print("Rows with all three features present:", check_non_missing.shape[0])
#filters out the 914 rows with missing feature values

#3. Normalize gender coding - map both datasets to the same, 0=female and 1=male
#PPMI: sex column is already 0=female, 1=male
check_non_missing["sex_ppmi_n"] = check_non_missing["SEX"]

check_non_missing["gen_clean"] = pd.to_numeric(check_non_missing["gen"], errors="coerce").astype("Int64")
#HIVE: gender colum is 1=male, 2=female --> map to 0/1
check_non_missing["sex_hive_n"] = check_non_missing["gen_clean"].map({1: 1, 2:0})
print(check_non_missing["sex_hive_n"].value_counts(dropna=False))


#4. PRepare race, the race codes are the same so just copying
check_non_missing["race_ppmi_n"] = check_non_missing["race_ppmi"].astype("Int64")

check_non_missing["race_hive_n"] = pd.to_numeric(check_non_missing["race_hive"], errors="coerce").astype("Int64")

check_non_missing["age_ppmi"] = pd.to_numeric(check_non_missing["age_ppmi"], errors="coerce")
check_non_missing["age_hive"] = pd.to_numeric(check_non_missing["age_hive"], errors="coerce")


#5. Define mismatch flags
#age (allows for +/= 1 year)
check_non_missing["age_mismatch"] = (
    (check_non_missing["age_ppmi"] - check_non_missing["age_hive"]).abs() > 1
)

#gender
check_non_missing["sex_mismatch"] = (
    check_non_missing["sex_ppmi_n"]!= check_non_missing["sex_hive_n"]
)

#race
check_non_missing["race_mismatch"] = (
    check_non_missing["race_ppmi_n"] != check_non_missing["race_hive_n"]
)

print("Age mismatches :", check_non_missing["age_mismatch"].sum())
print("Sex mismatches :", check_non_missing["sex_mismatch"].sum())
print("Race mismatches:", check_non_missing["race_mismatch"].sum())
#race mismatch is likely to be because of missing values that were shown earlier """

In [ ]:
import pandas as pd

# 0) Create a shared ID column name in BOTH datasets
PPMI_keep = PPMI_keep.copy()
hive_keep = hive_keep.copy()

PPMI_keep["_id"] = PPMI_keep["PATNO"].astype("string").str.strip()
hive_keep["_id"] = hive_keep["Individual"].astype("string").str.strip()

# (optional but recommended) ensure uniqueness before one_to_one merge
# If this fails, you still have duplicates per ID somewhere.
# print(PPMI_keep["_id"].duplicated().sum(), hive_keep["_id"].duplicated().sum())

# 1) Same cmp merge pattern as your old code (now with the shared _id)
cmp = (
    PPMI_keep[["_id", "age", "SEX", "race"]]
    .merge(
        hive_keep[["_id", "age", "gen", "race"]],
        on="_id",
        how="inner",
        suffixes=("_ppmi", "_hive"),
        validate="one_to_one"
    )
)

# 2) Ensure numeric for comparisons (safe even if you did it earlier)
cmp["age_ppmi"] = pd.to_numeric(cmp["age_ppmi"], errors="coerce")
cmp["age_hive"] = pd.to_numeric(cmp["age_hive"], errors="coerce")

cmp["race_ppmi"] = pd.to_numeric(cmp["race_ppmi"], errors="coerce").astype("Int64")
cmp["race_hive"] = pd.to_numeric(cmp["race_hive"], errors="coerce").astype("Int64")

# 3) Normalize sex/gender to 0=female, 1=male before comparing
# PPMI: SEX is 0/1 already
cmp["sex_ppmi_n"] = pd.to_numeric(cmp["SEX"], errors="coerce").astype("Int64")

# HIVE: gen is 1=male, 2=female -> map to 0/1
cmp["gen_clean"] = pd.to_numeric(cmp["gen"], errors="coerce").astype("Int64")
cmp["sex_hive_n"] = cmp["gen_clean"].map({1: 1, 2: 0}).astype("Int64")

# ---- mismatch flags (ignore missing on either side) ----
sex_mismatch = (
    cmp["sex_ppmi_n"].notna() & cmp["sex_hive_n"].notna() &
    (cmp["sex_ppmi_n"] != cmp["sex_hive_n"])
)

race_mismatch = (
    cmp["race_ppmi"].notna() & cmp["race_hive"].notna() &
    (cmp["race_ppmi"] != cmp["race_hive"])
)

tol = 1.0  # years
age_mismatch = (
    cmp["age_ppmi"].notna() & cmp["age_hive"].notna() &
    ((cmp["age_ppmi"] - cmp["age_hive"]).abs() > tol)
)

diff = cmp[sex_mismatch | race_mismatch | age_mismatch].copy()
diff["age_diff"] = (diff["age_ppmi"] - diff["age_hive"]).abs()

print("Rows merged:", len(cmp))
print("Rows with any mismatch:", len(diff))

diff[["_id",
      "SEX", "gen", "sex_ppmi_n", "sex_hive_n",
      "race_ppmi", "race_hive",
      "age_ppmi", "age_hive", "age_diff"]].head(20)


## plots


In [ ]:
#COHORT & CONCOHORT from PPMI dataset, (I think we are using CONCOHORT bc highlighted?)
#CONCOHORT 1=PD 2=HC 4=Prodromal       3=SWEDD (omit this)

#
df=final_df.copy()
label_map= {
    1: "PD",
    2: "HC",
    4: "Prodromal PD"
} 
counts = df[df["CONCOHORT"].isin([1, 2, 4])]["CONCOHORT"].value_counts().reindex([1,4,2])
x_labels= [label_map[i] for i in counts.index]

plt.figure()
plt.bar(x_labels, counts.values)
plt.xlabel("CONCOHORT")
plt.ylabel("Number of patients")
plt.title("Number of Patients per Concohort")
plt.show()


In [ ]:
df=final_df.copy()  #using df for making plots    #SEX column comes from PPMI dataset, "gen"=hive

cohort_map = {1: "PD", 2: "HC", 4: "Prodromal PD"}
sex_map    = {0: "Female", 1: "Male"}

df["CONCOHORT_lbl"] = df["CONCOHORT"].map(cohort_map)
df["SEX_lbl"] = df["SEX"].map(sex_map)

ct = (df[df["CONCOHORT"].isin([1,2,4])]
        .groupby(["CONCOHORT_lbl", "SEX_lbl"])
        .size()
        .unstack(fill_value=0)
        .reindex(["PD", "Prodromal PD", "HC"]))  # order

ax = ct.plot(kind="bar", stacked=True)
plt.title("Patients per Cohort (Sex)")
plt.xlabel("Concohort")
plt.ylabel("Number of patients")
plt.xticks(rotation=0)
plt.legend(title="Sex")
plt.tight_layout()
plt.show()

In [ ]:
cols_to_check = [
    "age",
    "SEX",
    "race", "COHORT", "CONCOHORT", "subgroup", "ageonset",
    "agediag", "updrs1_score", "updrs2_score", "updrs3_score", "updrs4_score",
    "updrs_totscore", "updrs_totscore_on",
    "LEDD", "age_DATSCAN", "moca", "MCI_testscores", 
    "SDMTOTAL", "TMT_A", "TMT_B", "VLTANIM", "rem",
    "upsit", "gds"

]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = final_df.copy()

# --- your existing labels ---
cohort_map = {1: "PD", 2: "HC", 4: "Prodromal PD"}
sex_map    = {0: "Female", 1: "Male"}

df["CONCOHORT_lbl"] = df["CONCOHORT"].map(cohort_map)
df["SEX_lbl"] = df["SEX"].map(sex_map)

# --- binning (example: AGE) ---
bins   = [0, 19, 29, 39, 49, 59, 69, 79, 100]
labels = ['0-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-100']

df["AGE_grp"] = pd.cut(df["age"], bins=bins, labels=labels, include_lowest=True)

# --- stacked counts: within each cohort, bars = age groups, stacked = sex (or outcome) ---
ct = (df[df["CONCOHORT"].isin([1,2,4])]
        .groupby(["CONCOHORT_lbl", "AGE_grp"])["SEX_lbl"]   # <-- change to 'Outcome' if you have it
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
     )

# Plot one cohort at a time (cleanest for readability)
for cohort in ["PD", "Prodromal PD", "HC"]:
    sub = ct[ct["CONCOHORT_lbl"] == cohort].set_index("AGE_grp").drop(columns=["CONCOHORT_lbl"])
    ax = sub.plot(kind="bar", stacked=True)
    plt.title(f"{cohort}: Patients by Age Group (stacked by Sex)")
    plt.xlabel("Age Group")
    plt.ylabel("Number of patients")
    plt.xticks(rotation=0)
    plt.legend(title="Sex")
    plt.tight_layout()
    plt.show()

# --- cleanup helper column(s) ---
df.drop(columns=["AGE_grp"], inplace=True)

In [ ]:
#missing values for: clinical severity, cognitive/neuropsychological measures
cols_to_check = [
    "updrs1_score", "updrs2_score", "updrs3_score", "updrs4_score",
    "updrs_totscore", "updrs_totscore_on",
    "LEDD", "age_DATSCAN", "moca", "MCI_testscores", 
    "SDMTOTAL", "TMT_A", "TMT_B", "VLTANIM", "rem",
    "upsit", "gds", "PRIMDIAG"

]

missing_values = final_df[cols_to_check].isna().sum()

print(missing_values)


missing_pct = final_df[cols_to_check].isna().mean() * 100

plt.figure()
missing_pct.plot(kind="bar", color = ['pink'])
plt.ylabel("Percentage missing (%)")
plt.title("Percentage of missing values per Variable (clinical severity, cognitive/neuropsycological measures)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
